# Transform Order Reviews Data
1. Filter out invalid rows i.e rows with review_id as null or duplicate review_id or invalid review_id (valid review_id is 32 character lenght alphanumeric string)
2. Filter out review with null order_id or invalid order_id 
3. Filter out rows with invalid review_score i.e if score is not between 1 and 5.
4. Write the transformed data to silver tables

In [0]:
#Imports
from pyspark.sql.functions import col

In [0]:
order_reviews_df=spark.read.table("olist_catalog.bronze.order_reviews")


### Step1 - Filter out invalid rows i.e rows with review_id as null or duplicate review_id or invalid review_id

In [0]:
order_reviews_valid1_df=(
    order_reviews_df.filter(
        col("review_id").isNotNull() & 
        col("review_id").rlike("^[A-Za-z0-9]{32}$")
    )
    .dropDuplicates(subset=["review_id"])       
)


### Step2 - Filter out review with null order_id or invalid order_id

In [0]:
order_reviews_valid2_df=(
    order_reviews_valid1_df.filter(col("order_id").isNotNull()& 
        col("order_id").rlike("^[A-Za-z0-9]{32}$"))
)

### Step3 - Filter out rows with invalid review_score i.e if score is not between 1 and 5.

In [0]:
order_reviews_final_df = (
    order_reviews_valid2_df.filter(
        col("review_score").isNotNull() & 
        (col("review_score") >= 1) & 
        (col("review_score") <= 5)
    )
)

### Step4 - Write the transformed data to silver tables

In [0]:
(
    order_reviews_final_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("olist_catalog.silver.order_reviews")
)

In [0]:
%sql
select * from olist_catalog.silver.order_reviews